In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import pandas as pd

def get_data(path):
    data = pd.read_csv(path)
    return data

epoch_better_path = "../../experiment_data/total_persistence/cross_epoch_better.csv"

epoch_data_better = get_data(epoch_better_path)

with open('../layer_orders_cross_layer.yml', 'r') as f:
    layer_order = yaml.safe_load(f)

with open('../layer_orders_intra_block.yml', 'r') as f:
    block_order = yaml.safe_load(f)
    
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

epoch_data_better["model"] = epoch_data_better["model"].apply(lambda x: f"{x}_better")
epoch_data_better.head()

,dataset,split,model,k,epoch,layer,thresh,thresh_mode,total_persistence,train_acc,val_acc
0,cifar,train,densenet_better,20,0,a1,0.0,0.0,140.614038,0.09992,0.0998
1,cifar,train,densenet_better,20,125,a1,0.0,0.0,4.806922,0.99836,0.8659
2,cifar,train,densenet_better,20,150,a1,0.0,0.0,2.526537,0.99990,0.8752
3,cifar,train,densenet_better,20,176,a1,0.0,0.0,7.426404,0.97020,0.8464
4,cifar,train,densenet_better,20,200,a1,0.0,0.0,8.803496,0.97902,0.8529


In [2]:
# epoch_trainUval = pd.concat([epoch_data[(epoch_data['split'] == 'trainUval')], epoch_data_better[(epoch_data_better['split'] == 'trainUval')]])
epoch_trainUval = pd.concat([epoch_data_better[(epoch_data_better['split'] == 'trainUval')]])
epoch_trainUval = epoch_trainUval.sort_values(by='epoch')

fig = make_subplots(specs=[[{"secondary_y": True}]], )

total_pers_trainUval = px.scatter(epoch_trainUval, x='epoch', y='total_persistence', color="dataset", color_discrete_sequence=color_seq, symbol="model")
train_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='train_acc', color="dataset", color_discrete_sequence=color_seq_train, symbol="model")
val_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='val_acc', color="dataset", color_discrete_sequence=color_seq_val, symbol="model", line_dash_sequence=['dash'])

for i in range(len(tuple(total_pers_trainUval.data))):
	# get dataset, model out of the trace
	dataset, model = str(total_pers_trainUval.data[i]['name']).split(", ")
    
	all_epochs = epoch_trainUval[(epoch_trainUval['dataset'] == dataset) & (epoch_trainUval['model'] == model)]
	# find the epoch where val acc is maximum
	best_epoch = all_epochs.sort_values(by='val_acc', ascending=False).iloc[0]['epoch']

	fig.add_trace(
		total_pers_trainUval.data[i],
		secondary_y=False,
	)

	fig.update_yaxes(secondary_y=False)

	fig.add_trace(
		train_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	fig.add_trace(
		val_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	# add a vertical line at best epoch
	fig.add_trace(go.Scatter(x=[best_epoch, best_epoch], y=[0, 1], mode="lines", line=dict(color=total_pers_trainUval.data[i]['marker']['color'], dash="dot"), legendgroup=total_pers_trainUval.data[i]['name'], showlegend=False), secondary_y=True)

# set title
fig.update_layout(
	title_text="Cross-Epoch on TrainUVal: Total Persistence vs Train/Val Accuracy"
)
fig.show()